In [4]:

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [6]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [7]:
arxiv.invoke("Attention is all you need?")

'Published: 2022-09-18\nTitle: The Mosquito incident and the need-(not-)to-know\nAuthors: Karl Svozil\nSummary: I am reporting here how professor Cristian Sorin Calude, henceforth called\nCris, became involved in the "Mosquito incident". More generally, the\nrelationship between the single individual vis-a-vis the state or collective is\nreviewed, with special emphasis to need-(not)-to-know secrets.\n\nPublished: 2025-06-12\nTitle: Needling Through the Threads: A Visualization Tool for Navigating Threaded'

In [8]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [9]:
wiki.invoke("What is LangGraph?")

'Page: List of unsolved problems in mathematics\nSummary: Many mathematical problems have been stated but not yet solved. These problems come from many areas of mathematics, such as theoretical physics, computer science, algebra, analysis, combinatorics, algebraic, differential, discrete and Euclidean geometries, graph theory, group theory, model theory, number theory, set theory, Ramsey theory, dynamical systems, and partial differential equations. Some problems belong to more than one discipline'

In [16]:
from dotenv import load_dotenv
load_dotenv()

import os

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [17]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

In [18]:
tavily.invoke("Provide a detailed overview of AI Agents")

"HTTPError('401 Client Error: Unauthorized for url: https://api.tavily.com/search')"

In [19]:
tools = [arxiv, wiki, tavily]

In [22]:
from langchain_groq import ChatGroq

# Use a supported model, e.g., "llama3-70b-8192"
llm = ChatGroq(model = "llama3-70b-8192")

llm_with_tools = llm.bind_tools(tools)

In [23]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage
llm_with_tools.invoke([HumanMessage(content=f"What is the recent AI News")])

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8gbfjyvz5', 'function': {'arguments': '{"query":"recent AI news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 1210, 'total_tokens': 1259, 'completion_time': 0.208302146, 'prompt_time': 0.039627069, 'queue_time': 0.274821031, 'total_time': 0.247929215}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_bf16903a67', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e338a382-e27d-4dec-8faf-d42b7007fd5c-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'recent AI news'}, 'id': '8gbfjyvz5', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1210, 'output_tokens': 49, 'total_tokens': 1259})

In [27]:
llm_with_tools.invoke([HumanMessage(content=f"What is the recent AI News")])

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hpch7r5hx', 'function': {'arguments': '{"query":"AI News"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 1210, 'total_tokens': 1258, 'completion_time': 0.192207923, 'prompt_time': 0.047216382, 'queue_time': 0.272313888, 'total_time': 0.239424305}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_bf16903a67', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--7d387bcc-43ab-4f6b-9233-d38c94daa82d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'AI News'}, 'id': 'hpch7r5hx', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1210, 'output_tokens': 48, 'total_tokens': 1258})

In [29]:
%pip install langgraph

from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

  Using cached langgraph_checkpoint-2.1.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached ormsgpack-1.10.0-cp312-cp312-win_amd64.whl.metadata (44 kB)
Using cached langgraph_checkpoint-2.1.1-py3-none-any.whl (43 kB)
Using cached ormsgpack-1.10.0-cp312-cp312-win_amd64.whl (121 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
### Entire Chatbot With LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

### Node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

# Build graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", END)


graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
messages=graph.invoke({"messages":HumanMessage(content="1706.03762")})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages=graph.invoke({"messages":HumanMessage(content="Provide me the top 10 recent AI news for MArch 3rd 2025")})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages=graph.invoke({"messages":HumanMessage(content="What is machine learning")})
for m in messages['messages']:
    m.pretty_print()